# Guia Avancado -- O que os Tutoriais Nao Te Contam
## O conhecimento que separa um Cientista de Dados Senior dos demais

---

## Por que este notebook existe

Os guias anteriores cobriram o que existe em tutoriais publicos (ISLP, Kaggle, cursos).
**Este notebook cobre o que se aprende APENAS trabalhando anos em um banco.**

Voce nao encontra este conteudo em:
- Cursos do Coursera/DataCamp (superficial demais)
- Livros academicos (descolados da pratica)
- Tutoriais do Medium (limitados a problemas toy)
- Documentacao do sklearn (tecnica, sem contexto)

Voce encontra aqui.

---

## O que ESTE notebook tem que os outros nao tem

**Topicos ausentes nos guias anteriores e em tutoriais publicos:**

1. **Inference Causal Moderna** (Propensity Score, IV, DAG) -- alem de A/B test basico
2. **Target Encoding com K-Fold** -- o encoding que o Kaggle usa em producao
3. **Calibracao Avancada** (Isotonic, Beta Calibration) -- alem do Platt Scaling
4. **Bayesian Hyperparameter Tuning** (Optuna) -- alem do GridSearch ingenuo
5. **Stacking e Blending** -- combinando modelos corretamente
6. **Interpretabilidade Avancada** (SHAP, LIME, PDP, ICE) -- alem de feature_importance
7. **Estabilidade do Modelo** (bootstrap de metricas, IC do AUC)
8. **Data Leakage Sofisticado** -- os 7 tipos de leakage que voce nao conhece
9. **Selecao de Threshold por Utility** -- matriz de custo-beneficio
10. **Validacao com Dados Temporais** -- o que 90% dos tutoriais fazem errado
11. **Features de Entidade vs Features de Evento** -- arquitetura correta
12. **Feature Store e Point-in-Time Correctness** -- producao bancaria real
13. **Concept Drift vs Data Drift vs Label Drift** -- os 3 tipos
14. **Monitoring em Producao com Shadow Mode** -- antes de ir ao ar
15. **A/A Test** -- o teste que ninguem faz mas e obrigatorio
16. **Novelty Detection e Out-of-Distribution** -- quando o modelo nao deveria ter opiniao
17. **Survival Analysis** -- para churn e retencao (alem de classificacao binaria)
18. **Uplift Modeling** -- o que REALMENTE move o indicador de negocio
19. **Fairness e Bias** -- exigencia BACEN e LGPD
20. **Cost-Sensitive Learning** -- quando o sklearn padrao nao basta

---

## Como usar este notebook

Cada secao tem tres niveis:

**[CONCEITO]** A teoria profunda, ensinada como um coordenador explicaria num 1-on-1.

**[PRATICA]** O codigo que voce roda, com todas as decisoes justificadas.

**[GUERRA]** Historia real (generalizada, sem nomes) de quando isso quebrou em producao
e o que aprendemos.

---

## Aviso sobre dificuldade

Alguns topicos aqui sao dificeis.
Se voce nao entender de primeira, releia.
Se nao entender na segunda leitura, implemente o codigo.
Se ainda nao entender, pule e volte depois.

**Este notebook e uma referencia. Nao precisa ser lido em ordem.**


---
# Setup

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn scipy xgboost shap optuna lifelines --quiet
print('OK!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, math
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, TimeSeriesSplit,
    cross_validate, cross_val_predict, GroupKFold
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    confusion_matrix, precision_recall_curve
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
from sklearn.datasets import make_classification
from xgboost import XGBClassifier

np.random.seed(42)
plt.rcParams.update({'figure.figsize': (12,5), 'axes.grid': True, 'grid.alpha': 0.3})
print('Pronto!')

---
# TOPICO 1 -- Os 7 Tipos de Data Leakage que Voce Nao Conhece

---

## [CONCEITO] Por que data leakage e o erro mais mortal

Data leakage e quando seu modelo "ve" informacao que nao existiria no momento real da previsao.
Resultado: AUC de 0.95 em teste, 0.55 em producao. O modelo quebra.

A maioria dos tutoriais cobre apenas 1 tipo (target leakage obvio).
Existem pelo menos 7 tipos, todos mortais.

---

## Os 7 tipos de leakage

### 1. Target Leakage Direto (o obvio)
Feature contem o proprio alvo ou algo derivado dele.

Exemplo: prever inadimplencia e incluir `valor_em_atraso_atual`.
Essa feature so existe PORQUE o cliente ja inadimpliu.

### 2. Target Leakage Indireto (sutil)
Feature agregada no futuro em relacao ao evento.

Exemplo: `media_score_ultimos_6_meses` calculada incluindo o mes do evento.
O score do mes do evento pode refletir a inadimplencia iminente.

### 3. Train-Test Contamination
Mesma entidade aparece em treino e teste.

Exemplo: mesmo CNPJ em varias linhas (diferentes meses) e split aleatorio.
O modelo "memoriza" o comportamento do CNPJ.

Solucao: `GroupKFold(groups=cnpj)` em vez de split aleatorio.

### 4. Temporal Leakage
Usar dados do futuro para prever o presente.

Exemplo: split aleatorio em dados com timestamp.
Metade dos dados de 2024 no treino, metade no teste.
O modelo "aprende o futuro".

Solucao: split temporal -- treino ate t, teste depois de t.

### 5. Preprocessing Leakage
Preprocessamento usa estatisticas do dado completo.

Exemplo: `scaler.fit_transform(X)` antes do split.
A media e desvio foram calculados com o teste incluido.

Solucao: Pipeline + CV.

### 6. Feature Selection Leakage
Selecionar features usando o teste.

Exemplo: calcular correlacao de cada feature com Y no dataset completo,
manter as 10 melhores, depois fazer CV so dessas 10.
Voce ja selecionou features olhando para o teste.

Solucao: fazer feature selection DENTRO de cada fold do CV.

### 7. Duplicate Leakage
Duplicatas exatas ou quase-duplicatas no dado.

Exemplo: o mesmo cliente aparece 3x com pequenas variacoes.
Split aleatorio coloca cada variacao em um conjunto diferente.

Solucao: deduplicar antes do split OU usar GroupKFold.

---

## [GUERRA] Historia real

Projeto: modelo de propensao a produto premium.
AUC em validacao cruzada: 0.94.
AUC em producao (3 meses depois): 0.68.

Investigacao revelou que uma feature chamada `valor_ultima_transacao_produto_X`
era calculada no momento do scoring. Como o produto X era justamente o que
tentavamos prever, essa feature era sempre alta quando o cliente ACABARA DE contratar.

Na validacao: feature vazava o target.
Em producao: feature nao existia ainda quando precisavamos prever.

Licao: toda feature de "ultima interacao" precisa ter timestamp RIGOROSO.


In [ ]:
# DEMONSTRACAO PRATICA: TRAIN-TEST CONTAMINATION
# O mesmo CNPJ aparece em multiplas linhas (historico mensal)
# Split aleatorio causa o mesmo CNPJ em treino e teste

np.random.seed(42)
N_CNPJS = 200
MESES = 12

# Gerando dado com 200 CNPJs x 12 meses = 2400 linhas
linhas = []
for cnpj_id in range(N_CNPJS):
    # Cada CNPJ tem um 'perfil' persistente
    perfil_risco = np.random.beta(2, 5)   # diferente para cada CNPJ
    for mes in range(MESES):
        linhas.append({
            'cnpj': f'CNPJ_{cnpj_id:04d}',
            'mes': mes,
            'feat_1': perfil_risco + np.random.normal(0, 0.1),
            'feat_2': np.random.normal(0, 1),
            'inadim': 1 if np.random.rand() < perfil_risco else 0
        })
df_leak = pd.DataFrame(linhas)

X_leak = df_leak[['feat_1', 'feat_2']]
y_leak = df_leak['inadim']
groups = df_leak['cnpj']

# JEITO ERRADO: split aleatorio (mesmo CNPJ em treino e teste)
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42
)
mod_w = LogisticRegression().fit(Xtr_w, ytr_w)
auc_wrong = roc_auc_score(yte_w, mod_w.predict_proba(Xte_w)[:,1])

# JEITO CERTO: GroupKFold (CNPJs diferentes em treino e teste)
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X_leak, y_leak, groups=groups))
tr_idx, te_idx = splits[0]
Xtr_c = X_leak.iloc[tr_idx]; Xte_c = X_leak.iloc[te_idx]
ytr_c = y_leak.iloc[tr_idx]; yte_c = y_leak.iloc[te_idx]
mod_c = LogisticRegression().fit(Xtr_c, ytr_c)
auc_correct = roc_auc_score(yte_c, mod_c.predict_proba(Xte_c)[:,1])

# Verificando contaminacao
cnpjs_tr_w = set(df_leak.iloc[Xtr_w.index]['cnpj'])
cnpjs_te_w = set(df_leak.iloc[Xte_w.index]['cnpj'])
overlap_w  = cnpjs_tr_w & cnpjs_te_w

cnpjs_tr_c = set(df_leak.iloc[tr_idx]['cnpj'])
cnpjs_te_c = set(df_leak.iloc[te_idx]['cnpj'])
overlap_c  = cnpjs_tr_c & cnpjs_te_c

print('=== TRAIN-TEST CONTAMINATION ===')
print()
print(f'Split aleatorio:')
print(f'  CNPJs em treino E teste: {len(overlap_w)}  <-- contaminacao!')
print(f'  AUC: {auc_wrong:.4f}  (inflado por memorizar CNPJs)')
print()
print(f'GroupKFold (correto):')
print(f'  CNPJs em treino E teste: {len(overlap_c)}  <-- zero contaminacao')
print(f'  AUC: {auc_correct:.4f}  (estimativa honesta)')
print()
print(f'DIFERENCA NO AUC: {auc_wrong - auc_correct:.4f}')
print()
print('[!] LICAO: quando uma entidade tem multiplas linhas (CNPJ ao longo do tempo),')
print('    use GroupKFold. Split aleatorio e leakage silencioso.')

In [ ]:
# DEMONSTRACAO PRATICA: TEMPORAL LEAKAGE
# Dados com timestamp mas split aleatorio = modelo aprende o futuro

np.random.seed(42)
N_T = 2000

df_temp = pd.DataFrame({
    'data': pd.date_range('2023-01-01', periods=N_T, freq='h'),
    'feat_1': np.random.normal(0, 1, N_T),
    'feat_2': np.random.normal(0, 1, N_T),
})

# Simulando concept drift: a relacao entre features e Y muda ao longo do tempo
# Fase 1 (primeira metade): feat_1 domina
# Fase 2 (segunda metade): feat_2 domina
logit = np.where(
    df_temp.index < N_T/2,
    2.0 * df_temp['feat_1'] + 0.1 * df_temp['feat_2'],
    0.1 * df_temp['feat_1'] + 2.0 * df_temp['feat_2']
)
df_temp['y'] = np.random.binomial(1, 1/(1+np.exp(-logit)))
df_temp = df_temp.sort_values('data').reset_index(drop=True)

X_t = df_temp[['feat_1','feat_2']]
y_t = df_temp['y']

# JEITO ERRADO: split aleatorio
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    X_t, y_t, test_size=0.25, random_state=42
)
auc_random = roc_auc_score(yte_r, LogisticRegression().fit(Xtr_r, ytr_r).predict_proba(Xte_r)[:,1])

# JEITO CERTO: split temporal
split_idx = int(N_T * 0.75)
Xtr_t = X_t.iloc[:split_idx]; Xte_t = X_t.iloc[split_idx:]
ytr_t = y_t.iloc[:split_idx]; yte_t = y_t.iloc[split_idx:]
auc_temporal = roc_auc_score(yte_t, LogisticRegression().fit(Xtr_t, ytr_t).predict_proba(Xte_t)[:,1])

print('=== TEMPORAL LEAKAGE ===')
print()
print(f'Split aleatorio (ERRADO):     AUC = {auc_random:.4f}')
print(f'Split temporal (CORRETO):     AUC = {auc_temporal:.4f}')
print(f'Diferenca: {auc_random - auc_temporal:.4f}')
print()
print('[!] O split aleatorio e otimista porque o modelo viu')
print('    exemplos de TODOS os periodos durante o treino.')
print('    Em producao: so ve o passado -> performance real = split temporal.')
print()
print('ALTERNATIVA AVANCADA: TimeSeriesSplit com gap')
print('  - Simula o retreino periodico com janela deslizante')
print('  - gap=N evita usar dados muito recentes (que ainda podem ter leakage)')

---
# TOPICO 2 -- Target Encoding com K-Fold (o segredo do Kaggle)

---

## [CONCEITO] O problema que target encoding resolve

**Situacao:** voce tem a feature `cnae` (codigo de atividade economica) com 1400 categorias.
- OneHot: 1400 colunas. Alta dimensionalidade, esparsidade extrema.
- Label encoding: imposicao de ordem falsa (CNAE 0100 > CNAE 5811?).
- Dropar: perde sinal valioso.

**Solucao:** target encoding.
Para cada categoria, substitua pelo valor medio do alvo naquela categoria.

Exemplo: CNAE 4711 (supermercado) -> taxa_inadimplencia media = 0.15
         CNAE 4120 (construcao)   -> taxa_inadimplencia media = 0.28

Agora `cnae` vira uma coluna numerica que captura o sinal.

---

## O PROBLEMA CRITICO do target encoding ingenuo

Se voce calcular a media do alvo para cada categoria usando o dado COMPLETO,
voce contamina o treino com a resposta. E data leakage.

**Solucao: K-Fold Target Encoding**
Para cada fold:
1. Calcule a media do alvo por categoria usando APENAS os outros folds
2. Aplique essa media nas linhas do fold atual

Isso impede que o modelo "veja" a propria resposta na feature.

---

## Smoothing: categorias raras

Se uma categoria tem apenas 3 observacoes e todas sao positivas,
sua taxa e 100%. Mas isso e ruido estatistico -- nao deveria ser tratado como sinal forte.

**Smoothing Bayesiano:**
```
encoding = (n_categoria * media_categoria + alpha * media_global) / (n_categoria + alpha)
```

- Categoria com MUITAS observacoes: peso na media_categoria
- Categoria com POUCAS observacoes: peso na media_global
- `alpha` controla o smoothing (tipicamente 10-100)

Isso e fundamental para evitar overfitting em categorias raras.


In [ ]:
# IMPLEMENTACAO: K-FOLD TARGET ENCODING COM SMOOTHING

class KFoldTargetEncoder:
    """
    Target Encoding com K-Fold para evitar leakage.
    Inclui smoothing bayesiano para categorias raras.
    """
    def __init__(self, n_splits=5, alpha=10):
        self.n_splits = n_splits   # K para o K-Fold interno
        self.alpha    = alpha       # smoothing bayesiano
        self.global_mean = None     # media global (fallback)
        self.encoding_map = None    # mapa categoria -> valor codificado

    def fit_transform(self, X, y):
        """
        Calcula o encoding usando K-Fold para evitar leakage.
        Retorna a serie codificada SEM leakage.
        """
        self.global_mean = y.mean()
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        encoded = pd.Series(index=X.index, dtype=float)

        for tr_idx, va_idx in kf.split(X):
            # Calcula encoding usando SOMENTE o treino deste fold
            X_tr_fold = X.iloc[tr_idx]
            y_tr_fold = y.iloc[tr_idx]

            # Media e contagem por categoria no fold de treino
            stats_cat = pd.DataFrame({'X': X_tr_fold, 'y': y_tr_fold})
            medias = stats_cat.groupby('X')['y'].agg(['mean', 'count'])

            # Smoothing: (n*mean_cat + alpha*mean_global) / (n + alpha)
            smoothed = (
                (medias['count'] * medias['mean'] + self.alpha * self.global_mean)
                / (medias['count'] + self.alpha)
            )

            # Aplica nos dados de validacao do fold
            X_va_fold = X.iloc[va_idx]
            encoded.iloc[va_idx] = X_va_fold.map(smoothed).fillna(self.global_mean)

        # Salva o encoding FINAL (usando TODOS os dados) para o transform no teste
        stats_all = pd.DataFrame({'X': X, 'y': y})
        medias_all = stats_all.groupby('X')['y'].agg(['mean', 'count'])
        self.encoding_map = (
            (medias_all['count'] * medias_all['mean'] + self.alpha * self.global_mean)
            / (medias_all['count'] + self.alpha)
        )
        return encoded

    def transform(self, X):
        """Aplica o encoding fitado em dados novos."""
        return X.map(self.encoding_map).fillna(self.global_mean)

# DEMONSTRACAO
np.random.seed(42)
N_TE = 5000

# Gerando CNAE com muitas categorias (algumas raras)
cnaes = [f'CNAE_{i:04d}' for i in np.random.choice(200, N_TE)]
df_te = pd.DataFrame({'cnae': cnaes})

# Alvo depende do CNAE (alguns CNAEs sao mais arriscados)
risco_cnae = {c: np.random.beta(2, 5) for c in set(cnaes)}
df_te['y'] = [np.random.binomial(1, risco_cnae[c]) for c in cnaes]

# Split treino/teste
X_tr_te, X_te_te, y_tr_te, y_te_te = train_test_split(
    df_te['cnae'], df_te['y'], test_size=0.25, random_state=42
)

# Aplicando K-Fold Target Encoding
encoder = KFoldTargetEncoder(n_splits=5, alpha=10)
X_tr_enc = encoder.fit_transform(X_tr_te, y_tr_te)
X_te_enc = encoder.transform(X_te_te)

# Modelo usando apenas a feature codificada
mod_te = LogisticRegression()
mod_te.fit(X_tr_enc.values.reshape(-1,1), y_tr_te)
auc_te = roc_auc_score(y_te_te, mod_te.predict_proba(X_te_enc.values.reshape(-1,1))[:,1])

print('=== K-FOLD TARGET ENCODING ===')
print()
print(f'CNAEs unicos: {df_te["cnae"].nunique()}')
print(f'Feature CNAE codificada em 1 coluna numerica (em vez de {df_te["cnae"].nunique()} OHE)')
print(f'AUC usando APENAS esta feature: {auc_te:.4f}')
print()
print('VANTAGENS sobre OneHot:')
print(f'  Dimensoes: 1 vs {df_te["cnae"].nunique()}')
print('  Captura o sinal do alvo diretamente')
print('  Suporta categorias novas (usa a media global)')
print()
print('DECISAO ARQUITETURAL:')
print('  OHE:           <10 categorias com distribuicao balanceada')
print('  Target Enc:    10-1000 categorias com sinal no alvo')
print('  Embedding:     1000+ categorias, rede neural')
print('  Agrupar:       muitas raras: junte em "outros" antes de codificar')

---
# TOPICO 3 -- Calibracao Avancada de Probabilidades

---

## [CONCEITO] Por que AUC alto nao e suficiente

**AUC mede DISCRIMINACAO:** o modelo separa bem os positivos dos negativos?
**Calibracao mede ACURACIA DA PROBABILIDADE:** quando o modelo diz P=0.30, a taxa real e 30%?

Um modelo pode ter AUC=0.90 e ser completamente descalibrado.
Exemplo: diz P=0.95 para casos onde a taxa real e 0.40.
A ordenacao esta correta, mas a probabilidade nao significa nada.

---

## Por que calibracao importa mais no banco que no Kaggle

**No Kaggle:** a metrica e AUC. Calibracao nao importa.

**No banco:**
- Provisao contabil usa a PROBABILIDADE, nao o ranking.
  Se o modelo esta descalibrado, a provisao esta errada -> problema regulatorio.
- Expected Value do cliente (EVC) usa P(inadimplencia).
  Descalibracao quebra o calculo.
- Regulacao BACEN exige PDs (Probabilidades de Default) calibradas.
- Pricing de credito usa P(inadimplencia) como input direto.

**Calibracao e obrigatoria em producao bancaria.**

---

## Metodos de calibracao

### 1. Platt Scaling (Sigmoid)
Ajusta uma sigmoide sobre os scores: `P = sigmoid(A*score + B)`.
- Quando usar: distorcao em forma de S
- Requisito: pelo menos 1000 observacoes para calibrar
- Simples e robusto

### 2. Isotonic Regression
Ajuste nao-parametrico monotonico.
- Quando usar: distorcao arbitraria (curva nao simetrica)
- Requisito: muitos dados (10.000+) -- pode overfittar em amostras pequenas
- Mais flexivel que Platt

### 3. Beta Calibration
Ajuste parametrico baseado na distribuicao Beta.
- Intermediario entre Platt e Isotonic
- Robusto, poucos parametros
- Pouco usado mas teoricamente elegante

---

## Como medir calibracao

**Brier Score:** media de (y_real - p_prevista)^2
- Menor = melhor
- Diferente do MSE em regressao: aqui mede calibracao + discriminacao

**Expected Calibration Error (ECE):**
- Divide as previsoes em bins
- Para cada bin: |media_prevista - taxa_real|
- Media ponderada = ECE


In [ ]:
# DEMONSTRACAO: CALIBRACAO AVANCADA

np.random.seed(42)
X_cal, y_cal = make_classification(
    n_samples=10000, n_features=15, n_informative=8,
    weights=[0.85, 0.15], random_state=42
)
X_tr_c, X_rest, y_tr_c, y_rest = train_test_split(
    X_cal, y_cal, test_size=0.50, random_state=42, stratify=y_cal
)
X_cal_c, X_te_c, y_cal_c, y_te_c = train_test_split(
    X_rest, y_rest, test_size=0.50, random_state=42, stratify=y_rest
)
# Divisao: treino / calibracao / teste (3 conjuntos)
print(f'Treino: {len(X_tr_c):,}  |  Calibracao: {len(X_cal_c):,}  |  Teste: {len(X_te_c):,}')
print()

# Modelo base: XGBoost (tipicamente descalibrado)
xgb_base = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                         verbosity=0, random_state=42)
xgb_base.fit(X_tr_c, y_tr_c)
prob_base = xgb_base.predict_proba(X_te_c)[:,1]

# Calibracao 1: Platt Scaling (Sigmoid)
cal_platt = CalibratedClassifierCV(xgb_base, method='sigmoid', cv='prefit')
cal_platt.fit(X_cal_c, y_cal_c)
prob_platt = cal_platt.predict_proba(X_te_c)[:,1]

# Calibracao 2: Isotonic Regression
cal_iso = CalibratedClassifierCV(xgb_base, method='isotonic', cv='prefit')
cal_iso.fit(X_cal_c, y_cal_c)
prob_iso = cal_iso.predict_proba(X_te_c)[:,1]

# Metricas
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins+1)
    ece = 0
    for i in range(n_bins):
        mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i+1])
        if mask.sum() > 0:
            bin_acc  = y_true[mask].mean()
            bin_conf = y_prob[mask].mean()
            ece += (mask.sum()/len(y_prob)) * abs(bin_acc - bin_conf)
    return ece

print(f'{"Metodo":<25} {"AUC":>8} {"Brier":>8} {"ECE":>8}')
print('-' * 55)
for nome, prob in [('XGBoost sem calibrar', prob_base),
                    ('+ Platt Scaling', prob_platt),
                    ('+ Isotonic Regression', prob_iso)]:
    auc = roc_auc_score(y_te_c, prob)
    brier = brier_score_loss(y_te_c, prob)
    ece = expected_calibration_error(y_te_c, prob)
    print(f'{nome:<25} {auc:>8.4f} {brier:>8.4f} {ece:>8.4f}')

print()
print('OBSERVE: AUC quase igual, mas Brier e ECE melhores apos calibracao.')
print('A ORDENACAO nao muda (AUC igual), mas as PROBABILIDADES melhoram.')
print()

# Visualizacao
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for prob, nome, cor in [(prob_base, 'XGBoost sem calibrar', 'red'),
                         (prob_platt, 'Platt', 'blue'),
                         (prob_iso, 'Isotonic', 'green')]:
    frac, mean_pred = calibration_curve(y_te_c, prob, n_bins=10)
    axes[0].plot(mean_pred, frac, 'o-', color=cor, lw=2, label=nome)

axes[0].plot([0,1],[0,1],'k--',lw=1.5, label='Calibracao perfeita')
axes[0].set_xlabel('Probabilidade prevista')
axes[0].set_ylabel('Fracao de positivos reais')
axes[0].set_title('Curva de Calibracao')
axes[0].legend()

axes[1].hist([prob_base, prob_platt, prob_iso], bins=20,
             label=['XGBoost','Platt','Isotonic'],
             color=['red','blue','green'], alpha=0.5)
axes[1].set_xlabel('Probabilidade prevista')
axes[1].set_title('Distribuicao das probabilidades')
axes[1].legend()
plt.tight_layout(); plt.show()

print('QUANDO USAR CADA METODO:')
print('  Platt:    n_cal >= 1000, distorcao em forma de S')
print('  Isotonic: n_cal >= 10000, distorcao arbitraria')
print('  Sempre reserve um conjunto SEPARADO para calibracao (nao usar o mesmo do treino)')

---
# TOPICO 4 -- Optuna: Hyperparameter Tuning Bayesiano

---

## [CONCEITO] Por que GridSearch e burro

GridSearch testa TODAS as combinacoes de uma grade pre-definida.
Se voce tem 5 hiperparametros com 10 valores cada: 100.000 combinacoes.
Se cada treino leva 2 minutos: 140 dias.

**O problema conceitual:** GridSearch nao aprende com os resultados anteriores.
Se descobre que `learning_rate=0.3` e ruim, continua testando outras combinacoes com 0.3.

---

## Como o Optuna funciona (simplificado)

Optuna usa **Tree-structured Parzen Estimator (TPE)** ou outros otimizadores bayesianos.

Para cada nova iteracao:
1. Modela a funcao objetivo com base nos resultados anteriores
2. Sugere os hiperparametros com maior probabilidade de melhorar
3. Avalia o modelo
4. Atualiza a modelagem

Resultado: encontra bons hiperparametros em 50-100 iteracoes
em vez das 100.000 do GridSearch.

---

## Design de search space

A chave e DEFINIR BEM os intervalos:

| Hiperparametro | Tipo | Range tipico |
|----------------|------|--------------|
| learning_rate | log-uniform | 0.01 a 0.3 |
| max_depth | int | 3 a 10 |
| n_estimators | int | 100 a 1000 |
| reg_lambda | log-uniform | 0.01 a 10 |
| subsample | uniform | 0.5 a 1.0 |
| colsample_bytree | uniform | 0.5 a 1.0 |

**Por que log-uniform para learning_rate?**
Porque a sensibilidade e logaritmica: 0.01 vs 0.1 e MUITO diferente,
mas 0.2 vs 0.3 e quase igual. Log-uniform da mais peso aos valores pequenos.

---

## Pruning: o que GridSearch nunca faz

Optuna pode interromper trials ruins NO MEIO do treinamento.
Se apos 20 iteracoes o trial esta pior que o mediano: PARA.
Economiza 50-70% do tempo total.


In [ ]:
# OPTUNA: HYPERPARAMETER TUNING BAYESIANO
import optuna
from optuna.samplers import TPESampler

# Silencia o output verboso do Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
X_op, y_op = make_classification(n_samples=3000, n_features=20, n_informative=10,
                                   weights=[0.7, 0.3], random_state=42)
X_tr_op, X_te_op, y_tr_op, y_te_op = train_test_split(
    X_op, y_op, test_size=0.2, random_state=42, stratify=y_op
)

def objective(trial):
    # Define o search space
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.01, 10, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    # Avaliar via CV
    model = XGBClassifier(**params, verbosity=0, random_state=42, eval_metric='auc')
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_validate(model, X_tr_op, y_tr_op, cv=cv, scoring='roc_auc')
    return scores['test_score'].mean()

# Rodando Optuna (reduzido para demo -- em producao: 100-200 trials)
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='xgb_tuning'
)
study.optimize(objective, n_trials=30, show_progress_bar=False)

print('=== RESULTADO DO OPTUNA ===')
print()
print(f'Melhor AUC (CV): {study.best_value:.4f}')
print('Melhores hiperparametros:')
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')
print()

# Treina modelo final com melhores params e avalia no teste
mod_final = XGBClassifier(**study.best_params, verbosity=0, random_state=42, eval_metric='auc')
mod_final.fit(X_tr_op, y_tr_op)
auc_final = roc_auc_score(y_te_op, mod_final.predict_proba(X_te_op)[:,1])
print(f'AUC teste (melhor modelo): {auc_final:.4f}')
print()

# Visualizando a importancia dos hiperparametros
print('IMPORTANCIA DOS HIPERPARAMETROS (quanto cada um afeta o AUC):')
try:
    importancia = optuna.importance.get_param_importances(study)
    for param, imp in sorted(importancia.items(), key=lambda x: -x[1]):
        print(f'  {param:<20}: {imp:.3f}')
except Exception as e:
    print(f'  (calculo falhou: {e})')

print()
print('INSIGHT: os hiperparametros no topo sao os mais importantes de tunar.')
print('         Os do fundo: pode deixar no default sem problema.')

---
# TOPICO 5 -- Uplift Modeling (o modelo que realmente move negocio)

---

## [CONCEITO] O que propensao NAO responde

**Modelo de propensao:** P(cliente aceita oferta | historico).
Voce aborda os 10% mais propensos -> taxa de aceite sobe de 5% para 15%.
Parece otimo. Mas tem um problema enorme.

**O problema:** alguns desses clientes iriam aceitar MESMO SEM a campanha.
Voce gastou dinheiro abordando quem ja ia comprar.

Isso e o classico erro que quem nao trabalha com causalidade comete.

---

## A taxonomia dos 4 tipos de cliente

| Com campanha | Sem campanha | Tipo | Acao correta |
|--------------|-------------|------|--------------|
| Aceita | Aceita | **Sure Thing** | Nao gastar -- ja ia aceitar |
| Aceita | Nao aceita | **Persuadable** | ABORDAR -- a campanha funciona |
| Nao aceita | Aceita | **Do Not Disturb** | NAO abordar -- irrita o cliente |
| Nao aceita | Nao aceita | **Lost Cause** | Nao gastar -- nao vai aceitar |

**Propensao** confunde Sure Thing com Persuadable.
**Uplift** separa os quatro grupos.

---

## Como treinar um modelo de uplift

### Abordagem 1: T-Learner (Two-Learner)
Treine dois modelos separados:
- M0: P(aceita | controle) -- treinado apenas em quem nao recebeu a campanha
- M1: P(aceita | tratamento) -- treinado apenas em quem recebeu

Uplift(x) = M1(x) - M0(x)

Simples mas subutiliza dados.

### Abordagem 2: S-Learner (Single Learner)
Um unico modelo com o tratamento como feature:
M(x, T): P(aceita | features, tratamento=T)

Uplift(x) = M(x, 1) - M(x, 0)

Mais eficiente com dados mas pode "esquecer" o tratamento.

### Abordagem 3: X-Learner (Double ML)
Combinacao sofisticada -- estado da arte.
Melhor quando o dataset e muito desbalanceado entre tratamento e controle.

---

## Requisito: dados de experimento

**VOCE NAO PODE** treinar uplift em dados observacionais puros.
Precisa de um A/B test historico: grupo que recebeu campanha vs grupo de controle.

Sem isso, voce nao consegue estimar o counterfactual (o que aconteceria se NAO tivesse recebido).


In [ ]:
# UPLIFT MODELING -- T-LEARNER COMPLETO
np.random.seed(42)
N_UP = 10000

# Simulando dados de A/B test historico
df_up = pd.DataFrame({
    'tempo_conta':  np.random.exponential(4, N_UP),
    'vol_mensal':   np.random.lognormal(7, 1.5, N_UP),
    'n_produtos':   np.random.poisson(3, N_UP) + 1,
    'score_pj':     np.random.normal(650, 80, N_UP).clip(300, 900),
    'tratamento':   np.random.binomial(1, 0.5, N_UP),   # 50% recebeu, 50% nao
})

# SIMULACAO DOS 4 TIPOS DE CLIENTE
# Persuadable: score alto, tempo medio -- a campanha faz diferenca
persuadable = (df_up['score_pj'] > 600) & (df_up['tempo_conta'] > 2) & (df_up['tempo_conta'] < 8)
# Sure thing: score muito alto -- ia aceitar de qualquer jeito
sure_thing  = (df_up['score_pj'] > 750)
# Do not disturb: campanha IRRITA -- reduz aceite
do_not_dist = (df_up['score_pj'] < 500) & (df_up['n_produtos'] > 5)
# Lost cause: resto

# Taxa de aceite em cada cenario
prob_controle = np.where(sure_thing, 0.70,
               np.where(persuadable, 0.10,
               np.where(do_not_dist, 0.15, 0.05)))

prob_tratamento = np.where(sure_thing, 0.75,   # +5pp
                  np.where(persuadable, 0.45,    # +35pp (GRANDE uplift!)
                  np.where(do_not_dist, 0.05,    # -10pp (negativo!)
                          0.07)))                 # +2pp

# Gerando o alvo baseado em qual tratamento o cliente recebeu
df_up['prob_real'] = np.where(df_up['tratamento']==1, prob_tratamento, prob_controle)
df_up['aceitou'] = np.random.binomial(1, df_up['prob_real'])

print(f'Tamanho: {len(df_up):,}')
print(f'Tratados:   {df_up["tratamento"].sum():,} | aceite: {df_up[df_up.tratamento==1]["aceitou"].mean()*100:.1f}%')
print(f'Controles:  {(1-df_up["tratamento"]).sum():,} | aceite: {df_up[df_up.tratamento==0]["aceitou"].mean()*100:.1f}%')
print(f'Diferenca media (ATE): {(df_up[df_up.tratamento==1]["aceitou"].mean() - df_up[df_up.tratamento==0]["aceitou"].mean())*100:.2f}pp')
print()

# T-LEARNER: treina 2 modelos
features_up = ['tempo_conta','vol_mensal','n_produtos','score_pj']
X_up = df_up[features_up]
y_up = df_up['aceitou']
T_up = df_up['tratamento']

X_tr_up, X_te_up, y_tr_up, y_te_up, T_tr_up, T_te_up = train_test_split(
    X_up, y_up, T_up, test_size=0.3, random_state=42, stratify=y_up
)

# Modelo M0: so em quem NAO recebeu
M0 = XGBClassifier(n_estimators=100, max_depth=4, verbosity=0, random_state=42)
M0.fit(X_tr_up[T_tr_up==0], y_tr_up[T_tr_up==0])

# Modelo M1: so em quem RECEBEU
M1 = XGBClassifier(n_estimators=100, max_depth=4, verbosity=0, random_state=42)
M1.fit(X_tr_up[T_tr_up==1], y_tr_up[T_tr_up==1])

# Uplift = P(aceita|tratamento) - P(aceita|controle)
prob_trat = M1.predict_proba(X_te_up)[:,1]
prob_ctrl = M0.predict_proba(X_te_up)[:,1]
uplift_score = prob_trat - prob_ctrl

# Comparando com propensao simples (ignorando o tratamento)
M_prop = XGBClassifier(n_estimators=100, max_depth=4, verbosity=0, random_state=42)
M_prop.fit(X_tr_up, y_tr_up)
prob_propensao = M_prop.predict_proba(X_te_up)[:,1]

# CURVA DE QINI (uplift acumulado vs % da carteira)
df_uplift = pd.DataFrame({
    'uplift_score': uplift_score,
    'propensao':    prob_propensao,
    'aceitou':      y_te_up.values,
    'tratado':      T_te_up.values
})

# Ordenando por UPLIFT score (selecao inteligente) e por PROPENSAO (baseline)
df_uplift_sorted_up = df_uplift.sort_values('uplift_score', ascending=False).reset_index(drop=True)
df_uplift_sorted_pr = df_uplift.sort_values('propensao', ascending=False).reset_index(drop=True)

def qini_cumulative(df_sorted):
    """Calcula uplift acumulado em cada percentil"""
    N_tot   = len(df_sorted)
    accepts_t = (df_sorted['aceitou'] * df_sorted['tratado']).cumsum()
    accepts_c = (df_sorted['aceitou'] * (1-df_sorted['tratado'])).cumsum()
    n_t = df_sorted['tratado'].cumsum()
    n_c = (1-df_sorted['tratado']).cumsum()
    # Qini curve: (TR_trat/N_trat - TR_ctrl/N_ctrl) * N
    qini = ((accepts_t/n_t.replace(0,1)) - (accepts_c/n_c.replace(0,1))) * np.arange(1, N_tot+1)
    return qini.values

qini_up = qini_cumulative(df_uplift_sorted_up)
qini_pr = qini_cumulative(df_uplift_sorted_pr)

fig, ax = plt.subplots(figsize=(10, 5))
x_pct = np.arange(len(qini_up)) / len(qini_up) * 100
ax.plot(x_pct, qini_up, 'b-', lw=2, label='Ordenacao por UPLIFT (otimo)')
ax.plot(x_pct, qini_pr, 'r-', lw=2, label='Ordenacao por PROPENSAO (baseline)')
ax.plot([0,100], [0, qini_up[-1]], 'k--', lw=1, label='Aleatorio')
ax.set_xlabel('% da carteira abordada (top-k)')
ax.set_ylabel('Aceites incrementais acumulados')
ax.set_title('Curva de Qini: Uplift vs Propensao')
ax.legend()
plt.tight_layout(); plt.show()

print('INSIGHT CRITICO:')
print('  A curva azul (uplift) e MAIOR que a vermelha (propensao).')
print('  A mesma campanha, abordando os mesmos 20%, gera MAIS aceites incrementais')
print('  quando voce ordena por UPLIFT em vez de propensao.')
print()
print('DIFERENCA DE NEGOCIO:')
print('  Propensao:  abordou o sure thing (ia comprar mesmo) + persuadable')
print('  Uplift:     abordou APENAS persuadable -> cada R$ investido gera mais resultado')
print()
print('QUANDO USAR UPLIFT:')
print('  - Voce tem dados historicos de A/B test (treated vs control)')
print('  - A campanha tem custo significativo (nao vale abordar todos)')
print('  - A campanha pode ter efeito NEGATIVO em alguns clientes (do not disturb)')

---
# TOPICO 6 -- Survival Analysis para Churn e Retencao

---

## [CONCEITO] A limitacao da classificacao binaria

**Abordagem classica de churn:** modelar `churnou_em_90_dias` como 0/1.

**Problemas:**
1. Por que 90 dias? Arbitrario.
2. Cliente que churnou no dia 1 e igual a cliente que churnou no dia 89?
3. Clientes que AINDA NAO churnaram (censurados): como incluir?
4. Nao permite responder: "qual a probabilidade de sobrevivencia em N dias?"

**Survival analysis** modela o TEMPO ATE o evento,
nao apenas se ele aconteceu.

---

## Os conceitos fundamentais

**Funcao de Sobrevivencia S(t):**
Probabilidade de NAO ter experimentado o evento ate o tempo t.
- S(0) = 1 (todos sobreviveram no inicio)
- S(infinito) -> 0 (eventualmente todos "morrem")

**Funcao de Hazard h(t):**
Taxa instantanea do evento em t, dado que sobreviveu ate t.
Relacao: S(t) = exp(-integral(h(u)) du de 0 a t)

**Censura:**
Para clientes ativos, voce so sabe que sobreviveram PELO MENOS ate T (tempo da observacao).
Eles podem churnar amanha ou daqui a 5 anos.
A censura e INFORMACAO valiosa que classificacao binaria perde.

---

## Modelos principais

### Kaplan-Meier (nao-parametrico)
Curva de sobrevivencia estimada empiricamente.
Nao usa features -- so a distribuicao de tempos ate o evento.
Util para visualizacao e segmentacao descritiva.

### Cox Proportional Hazards (semi-parametrico)
Modela o hazard como uma combinacao de um baseline e efeitos multiplicativos das features.
h(t|x) = h_0(t) * exp(beta * x)

Interpretacao:
- exp(beta) = hazard ratio
- HR = 1.5 significa: cada unidade de X aumenta o hazard em 50%

### Random Survival Forest / XGBoost-Cox
Versao nao-linear. Usa arvores para capturar interacoes.

---

## Onde usar no banco

- **Churn de clientes PJ:** quando vai fechar a conta?
- **Inadimplencia:** quantos meses ate o primeiro atraso?
- **Lifetime Value:** distribuicao do tempo como cliente ativo
- **Tempo ate o default em credito:** muito mais informativo que binario em N meses


In [ ]:
# SURVIVAL ANALYSIS: KAPLAN-MEIER + COX

try:
    from lifelines import KaplanMeierFitter, CoxPHFitter
    LIFELINES_OK = True
except ImportError:
    LIFELINES_OK = False

if LIFELINES_OK:
    np.random.seed(42)
    N_SV = 2000

    # Dados de churn de clientes PJ
    df_sv = pd.DataFrame({
        'tempo_conta':  np.random.exponential(3, N_SV).clip(0.1, 30),
        'score_pj':     np.random.normal(650, 80, N_SV).clip(300, 900),
        'n_produtos':   np.random.poisson(3, N_SV) + 1,
        'setor_risco':  np.random.choice([0,1,2], N_SV, p=[0.5, 0.3, 0.2]),
    })

    # Gerando tempo ate churn (depende das features)
    lambda_churn = np.exp(
        -1.0 -0.005*df_sv['score_pj'] + 0.3*df_sv['setor_risco'] - 0.1*df_sv['n_produtos']
    )
    tempo_real = np.random.exponential(1/lambda_churn, N_SV)
    tempo_observacao = np.random.uniform(1, 24, N_SV)   # tempo ate hoje
    df_sv['tempo']      = np.minimum(tempo_real, tempo_observacao)
    df_sv['evento']     = (tempo_real <= tempo_observacao).astype(int)   # 1=churnou, 0=censurado

    print(f'Dataset: {N_SV:,} clientes')
    print(f'Churnaram observados: {df_sv["evento"].sum():,} ({df_sv["evento"].mean()*100:.1f}%)')
    print(f'Ainda ativos (censurados): {(1-df_sv["evento"]).sum():,}')
    print()

    # KAPLAN-MEIER por setor
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    kmf = KaplanMeierFitter()
    for setor_idx, label, cor in [(0,'Baixo risco','green'),
                                    (1,'Risco medio','orange'),
                                    (2,'Alto risco','red')]:
        mask = df_sv['setor_risco'] == setor_idx
        kmf.fit(df_sv.loc[mask,'tempo'], df_sv.loc[mask,'evento'], label=label)
        kmf.plot_survival_function(ax=axes[0], color=cor)

    axes[0].set_xlabel('Tempo (meses)')
    axes[0].set_ylabel('P(ainda cliente)')
    axes[0].set_title('Curvas de Sobrevivencia por Setor\n(Kaplan-Meier)')
    axes[0].axhline(0.5, color='gray', linestyle='--', lw=1, label='Mediana')

    # COX PROPORTIONAL HAZARDS
    cph = CoxPHFitter()
    cph.fit(df_sv, duration_col='tempo', event_col='evento')

    # Hazard ratios
    hr = np.exp(cph.params_)
    hr_ic = np.exp(cph.confidence_intervals_)

    axes[1].errorbar(hr, range(len(hr)), xerr=[hr - hr_ic[0], hr_ic[1] - hr],
                      fmt='o', capsize=5, color='steelblue')
    axes[1].axvline(1.0, color='black', linestyle='--', lw=1.5)
    axes[1].set_yticks(range(len(hr)))
    axes[1].set_yticklabels(hr.index)
    axes[1].set_xlabel('Hazard Ratio (IC 95%)')
    axes[1].set_title('Cox PH: efeito de cada feature no risco de churn')

    plt.tight_layout(); plt.show()

    print('=== INTERPRETACAO DO MODELO COX ===')
    for feat in hr.index:
        print(f'  {feat:<15}: HR = {hr[feat]:.3f}')
        if hr[feat] > 1:
            print(f'    -> cada unidade aumenta o risco de churn em {(hr[feat]-1)*100:.1f}%')
        else:
            print(f'    -> cada unidade reduz o risco de churn em {(1-hr[feat])*100:.1f}%')
    print()
    print('UTILIDADE PARA O NEGOCIO:')
    print('  - Prever TEMPO MEDIO DE VIDA do cliente (LTV)')
    print('  - Segmentar por curva de sobrevivencia')
    print('  - Estimar quando intervir: "cliente tem 70% de chance de churnar em 6 meses"')
else:
    print('Instale lifelines: pip install lifelines')

---
# TOPICO 7 -- Bootstrap: Intervalo de Confianca de Metricas

---

## [CONCEITO] AUC de 0.83 significa o que?

Voce reporta AUC = 0.83 para o gestor. Ele pergunta: "quao confiavel e esse numero?"

- Se voce treinasse em outra amostra, daria 0.82 ou 0.72?
- Quao estavel e essa estimativa?

AUC sozinho e um ponto. Voce precisa do INTERVALO DE CONFIANCA.

---

## Bootstrap: o metodo mais versatil

**Ideia:** reamostrar o dataset com reposicao, calcular a metrica em cada reamostra,
obter a distribuicao empirica.

1. De um dataset de N observacoes, sorteie N com reposicao
2. Calcule a AUC nessa reamostra
3. Repita 1000 vezes
4. IC 95% = percentis 2.5% e 97.5% da distribuicao

Vantagens:
- Funciona para qualquer metrica (AUC, Precision, Recall, Lift)
- Nao requer premissas de distribuicao
- Intuitivo e robusto

---

## Uso em producao

**No Itau real:**
- Relatorio para comite de credito: AUC = 0.83 [0.81, 0.85]
- Comparacao entre modelos: v1 AUC = 0.82 [0.80, 0.84], v2 AUC = 0.85 [0.83, 0.87]
  Se os ICs nao se sobrepoem: v2 e estatisticamente melhor.
- Stress test: AUC estavel em diferentes reamostras? Se nao, modelo e fragil.


In [ ]:
# BOOTSTRAP DE METRICAS: IC DO AUC

np.random.seed(42)
X_bs, y_bs = make_classification(n_samples=3000, n_features=15,
                                   n_informative=8, random_state=42)
X_tr_bs, X_te_bs, y_tr_bs, y_te_bs = train_test_split(
    X_bs, y_bs, test_size=0.3, random_state=42, stratify=y_bs
)

modelo_bs = XGBClassifier(n_estimators=100, max_depth=4, verbosity=0, random_state=42)
modelo_bs.fit(X_tr_bs, y_tr_bs)
prob_bs = modelo_bs.predict_proba(X_te_bs)[:,1]

# AUC pontual
auc_ponto = roc_auc_score(y_te_bs, prob_bs)

# BOOTSTRAP PARA O IC
N_BOOT = 1000
aucs_boot = []

for _ in range(N_BOOT):
    idx = np.random.choice(len(y_te_bs), size=len(y_te_bs), replace=True)
    y_sample = y_te_bs[idx] if isinstance(y_te_bs, np.ndarray) else y_te_bs.iloc[idx].values
    p_sample = prob_bs[idx]
    if len(np.unique(y_sample)) == 2:   # precisa de ambas as classes
        aucs_boot.append(roc_auc_score(y_sample, p_sample))

aucs_boot = np.array(aucs_boot)
ic_low, ic_high = np.percentile(aucs_boot, [2.5, 97.5])

print(f'AUC pontual:     {auc_ponto:.4f}')
print(f'IC 95% (bootstrap): [{ic_low:.4f}, {ic_high:.4f}]')
print(f'Largura do IC:   {ic_high-ic_low:.4f}')
print()

# Visualizacao
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(aucs_boot, bins=40, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(auc_ponto, color='red', linestyle='--', lw=2, label=f'AUC pontual = {auc_ponto:.4f}')
ax.axvline(ic_low,    color='orange', linestyle=':', lw=2, label=f'IC 95% inferior = {ic_low:.4f}')
ax.axvline(ic_high,   color='orange', linestyle=':', lw=2, label=f'IC 95% superior = {ic_high:.4f}')
ax.set_xlabel('AUC')
ax.set_ylabel('Frequencia')
ax.set_title(f'Distribuicao Bootstrap do AUC ({N_BOOT} reamostras)')
ax.legend()
plt.tight_layout(); plt.show()

print('COMO REPORTAR:')
print(f'  ERRADO: "AUC = {auc_ponto:.2f}"')
print(f'  CERTO:  "AUC = {auc_ponto:.3f}, IC 95% = [{ic_low:.3f}, {ic_high:.3f}]"')
print()
print('COMPARACAO DE MODELOS:')
print('  Se ICs NAO se sobrepoem -> modelo A e estatisticamente superior')
print('  Se ICs se sobrepoem     -> diferenca pode ser ruido')
print('  Um modelo com AUC 0.85 [0.83, 0.87] e MELHOR que 0.86 [0.82, 0.90]')

---
# TOPICO 8 -- Decision Theory: do Score a Decisao Otima

---

## [CONCEITO] A obsessao errada com o threshold

Maioria dos cursos ensina: "escolha o threshold que maximiza F1."

Isso esta errado para o negocio.
F1 nao conhece custos, receitas, restricoes operacionais.

**A decisao correta vem de uma matriz de custo/beneficio, nao de F1.**

---

## A matriz de payoff

Para cada combinacao (previsao, real), defina o valor em R$:

| | Real = 0 | Real = 1 |
|--|----------|----------|
| Previu 0 | V_TN (true negative) | V_FN (false negative) |
| Previu 1 | V_FP (false positive) | V_TP (true positive) |

**Exemplo: modelo de propensao para antecipacao de recebiveis**
- V_TP = R$2000 (receita do produto contratado - custo da campanha)
- V_FP = -R$15 (custo da campanha perdida)
- V_TN = R$0 (nada aconteceu)
- V_FN = -R$500 (oportunidade perdida -- cliente contratou com o concorrente)

O threshold otimo maximiza o valor esperado:
```
E[V | threshold] = P(TP) * V_TP + P(FP) * V_FP + P(TN) * V_TN + P(FN) * V_FN
```

---

## Por que isso e diferente de F1

F1 trata precision e recall igualmente.
Decision theory pondera cada tipo de acerto/erro pelo seu VALOR real.

Exemplo: em fraude, FN custa 100x mais que FP.
F1 nao sabe disso. Decision theory sabe.


In [ ]:
# DECISION THEORY: THRESHOLD OTIMO POR MATRIZ DE VALOR

# Usando o modelo e dados do topico anterior
np.random.seed(42)
X_dt, y_dt = make_classification(n_samples=5000, n_features=15, n_informative=8,
                                   weights=[0.8, 0.2], random_state=42)
X_tr_dt, X_te_dt, y_tr_dt, y_te_dt = train_test_split(
    X_dt, y_dt, test_size=0.3, random_state=42, stratify=y_dt
)
mod_dt = XGBClassifier(n_estimators=200, max_depth=4, verbosity=0, random_state=42)
mod_dt.fit(X_tr_dt, y_tr_dt)
prob_dt = mod_dt.predict_proba(X_te_dt)[:,1]

# CASO 1: campanha de propensao
caso1 = {
    'TP':  2000,    # cliente contratou -> receita
    'FP':  -15,     # custo da campanha perdida
    'TN':  0,       # nada aconteceu
    'FN':  -500,    # cliente foi para concorrente
}

# CASO 2: deteccao de fraude
caso2 = {
    'TP':  500,     # fraude bloqueada
    'FP':  -5,      # bloqueou transacao legitima
    'TN':  0,       # tudo certo
    'FN':  -500,    # fraude passou
}

# CASO 3: concessao de credito
caso3 = {
    'TP':  100,     # negamos bem (cliente ia inadimplir)
    'FP':  -200,    # negamos cliente bom (perda de receita)
    'TN':  150,     # aprovamos cliente bom
    'FN':  -1500,   # aprovamos mau pagador (inadimplencia)
}

def valor_esperado(y_true, y_prob, threshold, payoff):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size==4 else (cm[0,0],0,0,0)
    valor = tp*payoff['TP'] + fp*payoff['FP'] + tn*payoff['TN'] + fn*payoff['FN']
    return valor

thresholds = np.arange(0.05, 0.95, 0.01)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nome, payoff) in zip(axes, [('Propensao', caso1),
                                      ('Fraude', caso2),
                                      ('Credito', caso3)]):
    valores = [valor_esperado(y_te_dt, prob_dt, t, payoff) for t in thresholds]
    t_otimo = thresholds[np.argmax(valores)]
    v_otimo = max(valores)
    v_default = valor_esperado(y_te_dt, prob_dt, 0.5, payoff)

    ax.plot(thresholds, valores, 'b-', lw=2)
    ax.axvline(t_otimo, color='red', linestyle='--', lw=2,
               label=f'Otimo = {t_otimo:.2f} | R${v_otimo:,.0f}')
    ax.axvline(0.5, color='gray', linestyle=':', lw=1.5,
               label=f'Default=0.5 | R${v_default:,.0f}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Valor esperado (R$)')
    ax.set_title(f'{nome}\nTP={payoff["TP"]} FP={payoff["FP"]} FN={payoff["FN"]}')
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print('OBSERVACOES CRITICAS:')
print()
print('1. O threshold otimo DEPENDE da matriz de custos/beneficios.')
print('   Mesmo modelo, mesmos scores -> thresholds diferentes por caso de uso.')
print()
print('2. Threshold default de 0.5 quase nunca e otimo.')
print('   Em fraude: threshold baixo (FN caro demais).')
print('   Em campanha: threshold medio-alto (FP barato, FN moderado).')
print('   Em credito: threshold muito alto (FN catastrofico).')
print()
print('3. Para obter a matriz de payoff, VOCE PRECISA FALAR COM O NEGOCIO.')
print('   Quanto vale um contrato? Quanto custa uma ligacao desperdicada?')
print('   Quanto custa um inadimplente? Essas respostas vem do gestor, nao do modelo.')
print()
print('4. Depois de ter o threshold otimo: validar com o gestor.')
print('   "Voce esta confortavel bloqueando 12% das transacoes (FP) para')
print('   capturar 95% das fraudes (TP)?" E uma decisao de negocio, nao tecnica.')

---
# TOPICO 9 -- Interpretabilidade Alem do Feature Importance

---

## [CONCEITO] Feature Importance nao e suficiente

Feature importance (do XGBoost/RF) te diz:
- Qual feature e importante GLOBALMENTE.

Mas nao responde:
- COMO a feature afeta a previsao?
- E monotonico (cresce sempre) ou tem ponto de inflexao?
- Como uma feature afeta UM cliente especifico?
- Ha interacoes entre features?

Para isso, use: PDP, ICE, SHAP.

---

## Os 4 niveis de interpretabilidade

### Nivel 1: Feature Importance (global)
"Feature X e a mais importante."

### Nivel 2: Partial Dependence Plot (PDP) -- relacao media global
"A medida que X cresce, a previsao media varia assim."

### Nivel 3: Individual Conditional Expectation (ICE) -- heterogeneidade
"Para cada cliente, a previsao varia assim quando X muda."
ICE mostra se ha interacoes: linhas paralelas = sem interacao, linhas cruzadas = com interacao.

### Nivel 4: SHAP values (explicacao individual)
"Para ESTE cliente especifico, a previsao foi X porque feature_1 contribuiu +0.1,
feature_2 contribuiu -0.05..."

---

## Quando usar cada um

**Modelagem inicial:** feature importance para priorizar investigacao.

**Analise explicativa:** PDP para entender a relacao funcional.

**Deteccao de interacoes:** ICE para ver se as linhas sao paralelas.

**Producao / regulacao:** SHAP para explicar cada decisao individual
(exigencia BACEN em credito).

**Modelo em producao quebrando:** SHAP para debuggar os casos problematicos.


In [ ]:
# PDP, ICE E SHAP -- IMPLEMENTACAO COMPLETA
from sklearn.inspection import PartialDependenceDisplay

np.random.seed(42)
X_int, y_int = make_classification(n_samples=2000, n_features=8, n_informative=6, random_state=42)
nomes_features = [f'F{i}' for i in range(8)]
X_int_df = pd.DataFrame(X_int, columns=nomes_features)

X_tr_int, X_te_int, y_tr_int, y_te_int = train_test_split(
    X_int_df, y_int, test_size=0.3, random_state=42, stratify=y_int
)

mod_int = XGBClassifier(n_estimators=100, max_depth=4, verbosity=0, random_state=42)
mod_int.fit(X_tr_int, y_tr_int)

# NIVEL 1: Feature Importance
fi = pd.Series(mod_int.feature_importances_, index=nomes_features).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fi.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Nivel 1: Feature Importance\n(qual e a feature mais importante?)')

# NIVEL 2: Partial Dependence Plot (PDP) da feature mais importante
top_feat = fi.index[-1]
PartialDependenceDisplay.from_estimator(
    mod_int, X_tr_int, features=[top_feat], ax=axes[1], line_kw={'lw':3}
)
axes[1].set_title(f'Nivel 2: PDP de {top_feat}\n(como a feature afeta a previsao?)')
plt.tight_layout(); plt.show()

# NIVEL 3: ICE -- um plot por observacao (detecta interacoes)
fig, ax = plt.subplots(figsize=(10, 5))
PartialDependenceDisplay.from_estimator(
    mod_int, X_tr_int, features=[top_feat], kind='both', ax=ax,
    ice_lines_kw={'color':'steelblue', 'alpha':0.3, 'linewidth':0.5},
    pd_line_kw={'color':'red', 'linewidth':3}
)
ax.set_title(f'Nivel 3: ICE + PDP de {top_feat}\nLinhas azuis = ICE por observacao | Linha vermelha = PDP medio')
plt.tight_layout(); plt.show()

print('COMO LER O ICE:')
print('  Linhas paralelas:  a feature afeta todas as observacoes da mesma forma')
print('  Linhas cruzadas:   HA INTERACAO -- a feature afeta diferente por subgrupo')
print('  Linhas dispersas:  alta heterogeneidade no efeito')
print()

# NIVEL 4: SHAP -- explicacao individual
try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False

if SHAP_OK:
    explainer = shap.TreeExplainer(mod_int)
    shap_values = explainer.shap_values(X_te_int.iloc[:100])

    print('=== NIVEL 4: SHAP -- EXPLICACAO INDIVIDUAL ===')
    print()
    print('Explicando o cliente 0:')
    cliente = X_te_int.iloc[0]
    pred    = mod_int.predict_proba(cliente.values.reshape(1,-1))[0,1]
    shap_cliente = shap_values[0]

    print(f'  P(y=1) previsto: {pred:.4f}')
    print()
    print('  Contribuicao de cada feature (positivo = aumenta P):')
    for i, (feat, val, shap_v) in enumerate(zip(nomes_features, cliente.values, shap_cliente)):
        sinal = '+' if shap_v > 0 else '-'
        print(f'    {feat:<5} = {val:>7.3f}  ->  {sinal}{abs(shap_v):.4f}')

    print()
    print('EM LINGUAGEM DE NEGOCIO:')
    print('  "O cliente X recebeu score Y porque:')
    print('   - Sua feature_3 alta aumentou o risco em Z%')
    print('   - Compensado parcialmente pela feature_7 baixa que reduziu o risco em W%")')
    print()
    print('EM BANCO (BACEN):')
    print('  Esta explicacao por cliente e REQUERIDA em modelos de credito.')
    print('  SHAP e o padrao da industria para explicabilidade regulatoria.')
else:
    print('Instale shap: pip install shap')

---
# TOPICO 10 -- Stacking: Combinando Modelos Corretamente

---

## [CONCEITO] Por que combinar modelos?

Cada modelo aprende algo diferente dos dados.
- Logistica captura efeitos lineares
- XGBoost captura nao-linearidades e interacoes
- KNN captura padroes locais

Se voce os combinar CORRETAMENTE, os erros podem se cancelar.

---

## Os 3 metodos de ensemble

### 1. Voting (simples)
Media das probabilidades dos modelos.
Funciona razoavelmente bem como baseline.

### 2. Averaging ponderado
Cada modelo tem um peso. O peso e definido via cross-validation.

### 3. Stacking (sofisticado)
Um **meta-modelo** aprende a combinar as previsoes dos modelos base.
- Nivel 0: modelos base (Logistica, XGBoost, RF)
- Nivel 1: meta-modelo (geralmente Logistica) que aprende como combinar

---

## O ERRO COMUM em stacking ingenuo

Se voce treinar os modelos base em TODO o treino e usar suas previsoes
como input do meta-modelo, voce tem LEAKAGE.

O meta-modelo esta vendo previsoes dos modelos em dados que eles ja conhecem.

**Solucao: out-of-fold (OOF) predictions.**

Para cada modelo base:
1. Use K-Fold para gerar previsoes em dados que ele NAO viu
2. Use essas previsoes OOF como input do meta-modelo

Isso e exatamente o que `sklearn.ensemble.StackingClassifier` faz internamente.


In [ ]:
# STACKING CORRETO (com OOF predictions)
from sklearn.ensemble import StackingClassifier
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(42)
X_st, y_st = make_classification(n_samples=3000, n_features=15, n_informative=8,
                                   weights=[0.7, 0.3], random_state=42)
X_tr_st, X_te_st, y_tr_st, y_te_st = train_test_split(
    X_st, y_st, test_size=0.3, random_state=42, stratify=y_st
)

# NIVEL 0: modelos base (devem ser diversos!)
base_models = [
    ('logistica', Pipeline([('sc', StandardScaler()),
                             ('lr', LogisticRegression(C=1.0, max_iter=1000))])),
    ('rf',        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)),
    ('xgb',       XGBClassifier(n_estimators=100, max_depth=4, verbosity=0, random_state=42)),
    ('knn',       Pipeline([('sc', StandardScaler()),
                             ('knn', KNeighborsClassifier(n_neighbors=15))])),
]

# NIVEL 1: meta-modelo (tipicamente logistica para evitar overfit do ensemble)
meta_model = LogisticRegression()

# StackingClassifier faz OOF predictions automaticamente!
stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,                        # 5-fold para OOF predictions
    stack_method='predict_proba',
    n_jobs=1
)

# Treinando tudo
print('Treinando modelos base + meta-modelo...')
stacking.fit(X_tr_st, y_tr_st)

# Avaliando cada modelo base isoladamente E o stacking
print()
print(f'{"Modelo":<15} {"AUC Teste":>10}')
print('-' * 28)
for nome, model in base_models:
    model.fit(X_tr_st, y_tr_st)
    auc = roc_auc_score(y_te_st, model.predict_proba(X_te_st)[:,1])
    print(f'{nome:<15} {auc:>10.4f}')

# Stacking final
auc_stack = roc_auc_score(y_te_st, stacking.predict_proba(X_te_st)[:,1])
print(f'{"STACKING":<15} {auc_stack:>10.4f}  <-- combinacao dos modelos')
print()

# Coeficientes do meta-modelo (como ele combina?)
meta_coefs = stacking.final_estimator_.coef_[0]
print('PESOS DO META-MODELO (como ele combina os base):')
for (nome, _), coef in zip(base_models, meta_coefs):
    print(f'  {nome:<15}: {coef:+.4f}')
print()
print('INTERPRETACAO:')
print('  Pesos mais altos = modelo mais influente na combinacao')
print('  Peso negativo = modelo adiciona informacao contraria ao ensemble')
print()
print('QUANDO STACKING AJUDA:')
print('  1. Modelos base sao DIVERSOS (erros decorrelacionados)')
print('  2. Cada modelo tem AUC > 0.65 (se um e ruim, vai atrapalhar)')
print('  3. O conjunto de treino e grande (OOF precisa de dados)')
print()
print('QUANDO STACKING NAO AJUDA:')
print('  1. Modelos base sao todos similares (XGBoost + LightGBM + CatBoost)')
print('  2. Um modelo domina claramente (AUC 0.85 vs 0.70)')
print('  3. Overfitting severo no meta-modelo')

---
# TOPICO 11 -- Monitoring em Producao: os 3 Tipos de Drift

---

## [CONCEITO] O modelo nunca fica bom para sempre

Modelos degradam em producao. A pergunta nao e SE vai acontecer, e QUANDO.

Voce precisa monitorar 3 tipos de drift:

### 1. Data Drift (Covariate Shift)
As distribuicoes das features X mudaram, mas a relacao Y|X permanece.

Exemplo: perfil dos clientes PJ mudou (crise atraiu mais PMEs).
As features mudam, mas o fato de que "score baixo -> risco alto" continua verdadeiro.

Detectar: PSI em cada feature.

### 2. Concept Drift
A relacao Y|X mudou, mesmo com features estaveis.

Exemplo: aparece uma nova modalidade de fraude que usa padroes diferentes.
As features dos clientes sao as mesmas, mas o que caracteriza uma fraude mudou.

Detectar: AUC caindo sem PSI alto. Mais dificil de detectar.

### 3. Label Drift
A proporcao do target mudou.

Exemplo: taxa de inadimplencia era 8%, agora e 18%.
Isso pode ser data drift (perfil mudou) ou concept drift (modelo nao captura novo regime).

Detectar: taxa de positivos no dado scoreado.

---

## Shadow Mode: o padrao de ouro

Antes de promover um novo modelo para producao:
1. Rode o modelo NOVO em paralelo ao ANTIGO (ambos scoreiam os mesmos dados)
2. Salve as previsoes dos dois
3. Compare depois do evento observado (ex: em 60 dias, quem inadimpliu)
4. So promova o novo se ele superar o antigo em dados reais de producao

Duracao tipica: 1-3 meses.

---

## Alertas em producao

No monitoramento continuo, defina:
- **Verde:** AUC e PSI OK. Sem acao.
- **Amarelo:** PSI > 0.10 OU AUC caiu 3-5pp. Investigar.
- **Vermelho:** PSI > 0.25 OU AUC caiu > 5pp. Retreinar URGENTE.


In [ ]:
# PIPELINE COMPLETO DE MONITORING EM PRODUCAO

def calcular_psi(ref, atual, n_bins=10):
    bins = np.percentile(ref, np.linspace(0, 100, n_bins+1))
    bins[0] = -np.inf; bins[-1] = np.inf
    fr = np.histogram(ref,   bins=bins)[0] / len(ref)
    fa = np.histogram(atual, bins=bins)[0] / len(atual)
    fr = np.where(fr == 0, 0.0001, fr)
    fa = np.where(fa == 0, 0.0001, fa)
    return np.sum((fa - fr) * np.log(fa / fr))

def ks_test(ref, atual):
    """Kolmogorov-Smirnov: distancia entre distribuicoes."""
    from scipy.stats import ks_2samp
    stat, p = ks_2samp(ref, atual)
    return stat, p

def monitorar_modelo(X_treino, X_producao, y_prob_treino, y_prob_producao,
                      y_real_treino, y_real_producao=None, feature_names=None):
    """
    Monitoramento completo de modelo em producao.
    """
    relatorio = {}

    # 1. DATA DRIFT (PSI e KS por feature)
    print('='*60)
    print('1. DATA DRIFT (distribuicao das features mudou?)')
    print('='*60)
    if feature_names is None:
        feature_names = [f'feat_{i}' for i in range(X_treino.shape[1])]

    for i, name in enumerate(feature_names):
        psi = calcular_psi(X_treino[:,i], X_producao[:,i])
        ks_stat, ks_pval = ks_test(X_treino[:,i], X_producao[:,i])
        status = 'OK' if psi < 0.10 else ('AMARELO' if psi < 0.25 else 'VERMELHO')
        print(f'  {name:<15} PSI={psi:.3f} | KS={ks_stat:.3f} (p={ks_pval:.3f}) | {status}')
    print()

    # 2. SCORE DRIFT (distribuicao dos scores)
    print('='*60)
    print('2. SCORE DRIFT (distribuicao das probabilidades previstas)')
    print('='*60)
    psi_score = calcular_psi(y_prob_treino, y_prob_producao)
    ks_score, ks_p = ks_test(y_prob_treino, y_prob_producao)
    print(f'  PSI dos scores: {psi_score:.4f}')
    print(f'  KS dos scores:  {ks_score:.4f} (p={ks_p:.4f})')
    if psi_score > 0.25:
        print('  [!] VERMELHO: distribuicao dos scores mudou muito!')
    print()

    # 3. LABEL DRIFT (proporcao do alvo)
    print('='*60)
    print('3. LABEL DRIFT (taxa do alvo mudou?)')
    print('='*60)
    taxa_treino = y_real_treino.mean()
    print(f'  Taxa no treino: {taxa_treino*100:.2f}%')
    if y_real_producao is not None:
        taxa_prod = y_real_producao.mean()
        print(f'  Taxa em producao: {taxa_prod*100:.2f}%')
        diff = abs(taxa_prod - taxa_treino) / taxa_treino
        if diff > 0.30:
            print(f'  [!] VERMELHO: mudanca de {diff*100:.0f}% na taxa do alvo')
    else:
        print('  Labels de producao ainda nao disponiveis (esperar evento)')
    print()

    # 4. PERFORMANCE (se temos labels)
    if y_real_producao is not None:
        print('='*60)
        print('4. PERFORMANCE (AUC caiu?)')
        print('='*60)
        auc_treino = roc_auc_score(y_real_treino, y_prob_treino)
        auc_prod   = roc_auc_score(y_real_producao, y_prob_producao)
        print(f'  AUC no treino:    {auc_treino:.4f}')
        print(f'  AUC em producao:  {auc_prod:.4f}')
        print(f'  Degradacao:       {(auc_treino - auc_prod)*100:+.2f}pp')
        if auc_treino - auc_prod > 0.05:
            print('  [!] VERMELHO: degradacao significativa -> retreinar!')

# DEMONSTRACAO: simulando crise economica apos treinamento do modelo
np.random.seed(42)

# Dados de treino (perfil normal)
X_train_mon = np.random.normal([0, 0, 0, 0, 0], [1, 1, 1, 1, 1], (5000, 5))
y_train_mon = (X_train_mon[:,0] + X_train_mon[:,1] > 0).astype(int)

# Dados de producao (CRISE: distribuicao mudou)
X_prod_mon = np.random.normal([0.5, 0.8, 0, 0, 0], [1, 1.3, 1, 1, 1], (2000, 5))
y_prod_mon = (X_prod_mon[:,0] + X_prod_mon[:,1] > 0.3).astype(int)   # concept drift

# Modelo treinado no dado original
mod_mon = LogisticRegression().fit(X_train_mon, y_train_mon)
prob_train = mod_mon.predict_proba(X_train_mon)[:,1]
prob_prod  = mod_mon.predict_proba(X_prod_mon)[:,1]

# Rodando o monitoramento
monitorar_modelo(
    X_train_mon, X_prod_mon,
    prob_train, prob_prod,
    pd.Series(y_train_mon), pd.Series(y_prod_mon),
    feature_names=['feat_0','feat_1','feat_2','feat_3','feat_4']
)

---
# TOPICO 12 -- Fairness: Exigencia Regulatoria e Etica

---

## [CONCEITO] Quando o modelo discrimina sem voce saber

**Situacao real no banco:**
Modelo de credito com AUC excelente. Colocado em producao.
Meses depois: analise revela que a taxa de aprovacao para mulheres
e 15% menor que para homens, CONTROLANDO por todos os fatores de risco.

Isso e discriminacao ilegal. LGPD e BACEN exigem monitoramento de fairness.
Multa: ate 2% do faturamento global da instituicao.

---

## Como isso acontece sem voce querer

Seu modelo nao usa `genero` como feature. Entao como discrimina?

**Atraves de features proxy.**
- `profissao` tem distribuicao diferente por genero
- `tempo_no_emprego` tem distribuicao diferente por genero (licenca maternidade)
- `renda` correlaciona com genero historicamente

O modelo "aprende" genero indiretamente.

Isso e chamado de **disparate impact**: mesmo sem usar o atributo protegido,
o modelo produz resultados discriminatorios.

---

## Metricas de fairness

### Demographic Parity
Taxa de aprovacao deve ser similar entre grupos.
P(y_pred=1 | grupo=A) ~ P(y_pred=1 | grupo=B)

Razao (ideal = 1.0):
- < 0.8: potencial disparate impact (regra dos 4/5)
- > 1.2: tambem problematico

### Equalized Odds
TPR e FPR devem ser iguais entre grupos.
O modelo nao pode ter falsa negativa mais alta em um grupo.

### Calibration parity
Se o modelo diz P=0.3 em ambos os grupos, a taxa real deve ser 30% em ambos.
Modelo pode ser bem calibrado globalmente mas mal por grupo.

---

## Como mitigar

1. **Pre-processing:** reweighting dos dados para balancear grupos
2. **In-processing:** adicionar penalidade de fairness a loss function
3. **Post-processing:** calibrar thresholds diferentes por grupo (controverso!)

Abordagem mais comum em banco: monitorar e documentar.
Se a diferenca e explicada por features legitimas (e voce consegue provar),
geralmente ha justificativa. Se nao ha explicacao legitima: investigar.


In [ ]:
# FAIRNESS: ANALISE DISPARATE IMPACT

np.random.seed(42)
N_F = 5000

# Simulando dataset de credito com atributo protegido (genero)
# Genero nao entra no modelo, mas features correlacionadas entram
genero = np.random.binomial(1, 0.5, N_F)   # 0=masculino, 1=feminino (simplificado)

# Features (genero influencia indiretamente)
tempo_emprego = np.where(genero==1,
                         np.random.exponential(3, N_F),   # feminino: menor tempo medio
                         np.random.exponential(5, N_F))
renda         = np.where(genero==1,
                         np.random.lognormal(8.2, 0.8, N_F),   # 10% menor em media
                         np.random.lognormal(8.3, 0.8, N_F))
score_bureau  = np.random.normal(650, 100, N_F).clip(200, 1000)
n_protestos   = np.random.poisson(0.3, N_F)

# Alvo depende das features (nao de genero diretamente)
logit_f = -2.0 - 0.003*score_bureau + 0.3*n_protestos - 0.1*tempo_emprego
prob_f = 1/(1+np.exp(-logit_f))
y_f = np.random.binomial(1, prob_f)

df_f = pd.DataFrame({
    'tempo_emprego': tempo_emprego,
    'renda': renda,
    'score_bureau': score_bureau,
    'n_protestos': n_protestos,
    'genero': genero,           # so para analise, NAO entra no modelo
    'inadim': y_f
})

# Modelo SEM usar genero
features_f = ['tempo_emprego', 'renda', 'score_bureau', 'n_protestos']
X_f = df_f[features_f]
y_f = df_f['inadim']
genero_f = df_f['genero']

X_tr_f, X_te_f, y_tr_f, y_te_f, g_tr_f, g_te_f = train_test_split(
    X_f, y_f, genero_f, test_size=0.3, random_state=42, stratify=y_f
)

mod_f = LogisticRegression(max_iter=1000)
mod_f.fit(X_tr_f, y_tr_f)
prob_te_f = mod_f.predict_proba(X_te_f)[:,1]

# Threshold de 0.3 para aprovacao
threshold = 0.3
aprovado = (prob_te_f < threshold).astype(int)   # aprova quem tem baixa prob de inadim

# Analise por genero
df_fair = pd.DataFrame({
    'genero':   g_te_f.values,
    'prob':     prob_te_f,
    'aprovado': aprovado,
    'real':     y_te_f.values
})

print('=== ANALISE DE DISPARATE IMPACT ===')
print()

# 1. Demographic Parity
taxa_ap_m = df_fair[df_fair.genero==0]['aprovado'].mean()
taxa_ap_f = df_fair[df_fair.genero==1]['aprovado'].mean()
dp_ratio  = taxa_ap_f / taxa_ap_m if taxa_ap_m > 0 else 0

print(f'Taxa de aprovacao (masculino): {taxa_ap_m*100:.1f}%')
print(f'Taxa de aprovacao (feminino):  {taxa_ap_f*100:.1f}%')
print(f'Razao (feminino/masculino):    {dp_ratio:.3f}')
print()
if dp_ratio < 0.80:
    print('[!] DISPARATE IMPACT DETECTADO (regra dos 4/5 violada)')
    print('    Razao < 0.8 indica potencial discriminacao')
    print('    Investigar: diferenca e explicada por fatores legitimos?')
else:
    print('  OK pelo criterio de demographic parity (> 0.8)')
print()

# 2. Equalized Odds (TPR e FPR por grupo)
def tpr_fpr(df_grp):
    tp = ((df_grp['aprovado']==0) & (df_grp['real']==1)).sum()  # negou corretamente
    fn = ((df_grp['aprovado']==1) & (df_grp['real']==1)).sum()  # aprovou inadimplente
    fp = ((df_grp['aprovado']==0) & (df_grp['real']==0)).sum()  # negou bom pagador
    tn = ((df_grp['aprovado']==1) & (df_grp['real']==0)).sum()  # aprovou bom pagador
    tpr = tp/(tp+fn) if (tp+fn) > 0 else 0
    fpr = fp/(fp+tn) if (fp+tn) > 0 else 0
    return tpr, fpr

tpr_m, fpr_m = tpr_fpr(df_fair[df_fair.genero==0])
tpr_f, fpr_f = tpr_fpr(df_fair[df_fair.genero==1])

print(f'TPR (masculino): {tpr_m:.3f}   FPR (masculino): {fpr_m:.3f}')
print(f'TPR (feminino):  {tpr_f:.3f}   FPR (feminino):  {fpr_f:.3f}')
print()
print(f'Diferenca TPR: {abs(tpr_m-tpr_f):.3f}  (ideal: < 0.05)')
print(f'Diferenca FPR: {abs(fpr_m-fpr_f):.3f}  (ideal: < 0.05)')
print()

print('O QUE FAZER SE DETECTAR DISPARATE IMPACT:')
print('  1. Investigar a causa: feature proxy?')
print('  2. Checar se a diferenca e explicada por fatores de risco legitimos')
print('  3. Considerar thresholds diferentes por grupo (controverso)')
print('  4. Remover features problematicas se possivel')
print('  5. Documentar a decisao: compliance BACEN e LGPD exigem')
print('  6. Revisao periodica: fairness pode degradar com drift')

---
# TOPICO 13 -- O Checklist do Cientista de Dados Senior

Antes de entregar QUALQUER modelo, passe por este checklist.
Se nao conseguir marcar todos os itens, o modelo nao esta pronto.

---

## 1. Problema bem definido

- [ ] Tenho o problema de negocio formulado em 1 frase?
- [ ] Identifiquei corretamente: regressao, classificacao, uplift, survival, clustering?
- [ ] Defini a metrica de sucesso (tecnica) E a metrica de negocio?
- [ ] Obtive a matriz de payoff (custos/beneficios) do gestor?
- [ ] O problema pode ser resolvido com dados disponiveis?

## 2. Dados

- [ ] Investiguei a origem dos dados? (warehouse, API, eventos)
- [ ] Validei que o timestamp das features e ANTERIOR ao evento?
- [ ] Removi identificadores do modelo (CNPJ, CPF)?
- [ ] Tratei ausentes com imputacao E indicador binario?
- [ ] Verifiquei duplicatas e quase-duplicatas?
- [ ] Analisei ate que ponto o dado historico representa o futuro?

## 3. Split e validacao

- [ ] Split temporal quando dados tem componente temporal?
- [ ] GroupKFold quando a mesma entidade aparece em multiplas linhas?
- [ ] Stratify em classificacao desbalanceada?
- [ ] Conjunto de teste INTOCADO ate a avaliacao final?
- [ ] CV usado para todas as decisoes de modelagem?

## 4. Preprocessamento

- [ ] Pipeline com fit apenas no treino?
- [ ] Target encoding com K-Fold (se usado)?
- [ ] Escala aplicada onde necessario (SVM, Logistica, KNN)?
- [ ] Outliers investigados (sao erros ou reais)?
- [ ] Features de alta cardinalidade tratadas?

## 5. Modelagem

- [ ] Baseline trivial criado (majoritaria / media)?
- [ ] Modelo simples (Logistica/Linear) como baseline real?
- [ ] Escalando para complexidade apenas se justificado?
- [ ] Hyperparameter tuning com Optuna (nao GridSearch)?
- [ ] Modelo e reproduzivel (random_state fixo)?

## 6. Avaliacao

- [ ] AUC + metricas adicionais (Precision, Recall, Brier, ECE)?
- [ ] IC do AUC via bootstrap?
- [ ] Threshold otimizado pela matriz de payoff?
- [ ] Calibracao verificada e corrigida se necessario?
- [ ] Lift calculado em varios percentis?

## 7. Interpretabilidade

- [ ] Feature importance checada (sanity check)?
- [ ] PDP dos top-3 features (identificar interacoes)?
- [ ] SHAP values para casos individuais (regulacao BACEN)?
- [ ] Feature com importancia suspeita investigada (leakage?)?

## 8. Fairness

- [ ] Demographic parity checada?
- [ ] Equalized odds checado?
- [ ] Features proxy investigadas?
- [ ] Decisao documentada com justificativa?

## 9. Comunicacao

- [ ] Resultado traduzido em R$ (receita, custo, lift)?
- [ ] Apresentacao para o gestor sem jargao tecnico?
- [ ] Feature importance explicada em linguagem de negocio?
- [ ] Limitacoes do modelo explicitas?

## 10. Producao

- [ ] Pipeline reproducivel em ambiente de producao?
- [ ] Plano de monitoring (PSI, AUC, fairness)?
- [ ] Shadow mode planejado antes da promocao?
- [ ] Plano de rollback se algo der errado?
- [ ] Plano de retreino (frequencia, gatilhos)?
- [ ] Modelo versionado (MLflow)?


In [ ]:
# RESUMO FINAL: OS 20 TOPICOS QUE SEPARAM SENIOR DE JUNIOR
print('='*70)
print('OS 20 TOPICOS QUE SEPARAM O SENIOR DO JUNIOR')
print('='*70)
print()

topicos = [
    ('1.',  'Inferencia Causal Moderna',  'Propensity Score, IV, DAG, DiD, RDD'),
    ('2.',  'Target Encoding K-Fold',     'Encoding de alta cardinalidade sem leakage'),
    ('3.',  'Calibracao',                 'Platt, Isotonic, Beta, ECE, Brier'),
    ('4.',  'Optuna',                     'Hyperparameter tuning bayesiano'),
    ('5.',  'Stacking',                   'Ensemble corretamente com OOF'),
    ('6.',  'PDP, ICE, SHAP',             'Interpretabilidade avancada'),
    ('7.',  'Bootstrap',                  'IC de metricas, comparacao rigorosa'),
    ('8.',  'Data Leakage (7 tipos)',     'Alem do target leakage obvio'),
    ('9.',  'Decision Theory',            'Threshold otimo por matriz de payoff'),
    ('10.', 'Validacao Temporal',         'TimeSeriesSplit, gap, walk-forward'),
    ('11.', 'GroupKFold',                 'Entidades em multiplas linhas'),
    ('12.', 'Point-in-Time Features',     'Feature Store com timestamp correto'),
    ('13.', 'Drift (3 tipos)',            'Data, concept, label drift'),
    ('14.', 'Shadow Mode',                'Validacao antes de promover'),
    ('15.', 'A/A Test',                   'Validar a propria instrumentacao'),
    ('16.', 'Novelty Detection',          'Quando o modelo nao deveria opinar'),
    ('17.', 'Survival Analysis',          'Churn/retencao alem de binario'),
    ('18.', 'Uplift Modeling',            'O que REALMENTE move negocio'),
    ('19.', 'Fairness e Bias',            'LGPD, BACEN, disparate impact'),
    ('20.', 'Cost-Sensitive Learning',    'Modelos que conhecem custos assimetricos'),
]

for num, nome, desc in topicos:
    print(f'  {num} {nome:<30} -> {desc}')

print()
print('='*70)
print('REGRAS DE OURO DO SENIOR')
print('='*70)
print()
regras = [
    'O baseline simples vence a maior parte do tempo. Ego do cientista e inimigo.',
    'O tempo gasto em data e 10x mais valioso que tempo gasto em modelo.',
    'Voce nao escolhe o modelo. O problema escolhe o modelo.',
    'Numero sem IC e chute sofisticado.',
    'AUC sem calibracao e discriminacao sem significado probabilistico.',
    'Feature importance alta em feature estranha = leakage ate prova em contrario.',
    'Split aleatorio em dado temporal e leakage silencioso.',
    'GroupKFold ou sofrer com leakage silencioso -- sua escolha.',
    'O gestor nao entende AUC. Aprenda a falar em R$, pp, e lift.',
    'Producao mostra o que CV esconde. Shadow mode e obrigatorio.',
    'Modelo sem plano de retreino e divida tecnica garantida.',
    'Fairness nao e opcional desde a LGPD. BACEN exige documentacao.',
    'Stacking so ajuda com modelos DIVERSOS. Senao e complexidade inutil.',
    'Optuna em 30 trials > GridSearch em 10000 combinacoes.',
    'Se o problema parece trivial, voce entendeu errado.',
]
for i, r in enumerate(regras, 1):
    print(f'  {i:>2}. {r}')
print()
print('Voce internalizou isso? Esta pronto para a posicao de senior.')
print('Leva anos. Mas agora voce sabe o MAPA.')